# Weekly Time Index Construction and Gap Analysis

Epidemiological surveillance data often contain missing observations due to reporting delays, changes in surveillance systems, or country-specific coverage limitations.

In this notebook, we:
- Construct a **weekly calendar** aligned across countries
- Reindex each country’s series to this calendar
- Identify and characterize **missing data gaps**

This analysis is used to decide which countries can be reliably modeled and to define appropriate imputation strategies.


In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent   # go up one level
sys.path.insert(0, str(PROJECT_ROOT))


In [12]:
from src.gaps import gap_summary

In [13]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR.parent / "data"

ari_df = pd.read_csv(DATA_DIR / "latest-ARI_incidence.csv")
ili_df = pd.read_csv(DATA_DIR / "latest-ILI_incidence.csv")

# 1) convert dates to datetime
ari_df["truth_date"] = pd.to_datetime(ari_df["truth_date"])
ili_df["truth_date"] = pd.to_datetime(ili_df["truth_date"])

# 2) order
ari_df = ari_df.sort_values(["location", "truth_date"]).reset_index(drop=True)
ili_df = ili_df.sort_values(["location", "truth_date"]).reset_index(drop=True)

# 3) create weekly calendar using datetimes
full_weeks = pd.date_range(
    start=ari_df["truth_date"].min(),
    end=ari_df["truth_date"].max(),
    freq="W-SUN"
)

# 4) Reindex for BE
be_ari = ari_df[ari_df["location"] == "BE"].set_index("truth_date").sort_index()
be_ari_full = be_ari.reindex(full_weeks)

print(be_ari_full.head(10))
print(be_ari_full.tail(10))


           location year_week   value
2014-10-05       BE  2014-W40  1407.5
2014-10-12       BE  2014-W41  1493.7
2014-10-19       BE  2014-W42  1441.3
2014-10-26       BE  2014-W43  1514.0
2014-11-02       BE  2014-W44  1401.2
2014-11-09       BE  2014-W45  1560.0
2014-11-16       BE  2014-W46  1392.1
2014-11-23       BE  2014-W47  1748.8
2014-11-30       BE  2014-W48  1904.9
2014-12-07       BE  2014-W49  2036.4
           location year_week   value
2024-08-11       BE  2024-W32   419.6
2024-08-18       BE  2024-W33   307.9
2024-08-25       BE  2024-W34   394.7
2024-09-01       BE  2024-W35   417.6
2024-09-08       BE  2024-W36   713.6
2024-09-15       BE  2024-W37   927.0
2024-09-22       BE  2024-W38   993.0
2024-09-29       BE  2024-W39  1216.6
2024-10-06       BE  2024-W40  1182.0
2024-10-13       BE  2024-W41  1384.1


In [15]:
be_ari_full["value"].isna().mean()

np.float64(0.0)

In [16]:
be_ari_full["value"].notna().sum(), len(be_ari_full)

(np.int64(524), 524)

In [17]:
is_missing = be_ari_full["value"].isna()

# lenghts of consecutive NaN blocks
gap_lengths = (
    is_missing.astype(int)
    .groupby((is_missing != is_missing.shift()).cumsum())
    .sum()
)

# only keep gap blocks (NaN)
gap_lengths = gap_lengths[gap_lengths > 0]

gap_lengths.tolist()[:20], gap_lengths.max()

([], nan)

## Identification and characterization of missing data gaps

For each country, the weekly series is reindexed to a common calendar and missing values are identified.

We summarize missing data patterns using:
- Total number of missing weeks
- Number of consecutive missing blocks (gaps)
- Length of the largest gap
- Temporal location of gaps

This information is later used to:
- Filter countries with excessive missing data
- Choose suitable imputation methods


In [18]:
ari_gap_cov = gap_summary(ari_df, "ARI", calendar_mode="fixed_2014")
ili_gap_cov = gap_summary(ili_df, "ILI", calendar_mode="fixed_2014")

ari_gap_in = gap_summary(ari_df, "ARI", calendar_mode="data_range")
ili_gap_in = gap_summary(ili_df, "ILI", calendar_mode="data_range")

gap_all_cov = pd.concat([ari_gap_cov, ili_gap_cov], ignore_index=True)
gap_all_in  = pd.concat([ari_gap_in,  ili_gap_in],  ignore_index=True)


In [19]:
gap_all_in.sort_values("max_gap_length", ascending=False).head(15)

,disease,location,calendar_mode,n_weeks,n_observed,n_missing,n_gaps,gap_lengths,gap_starts,max_gap_length,biggest_gap_start,last_gap_start
13,ARI,MT,data_range,524,30,494,4,"[1, 1, 1, 491]","[2014-10-05 00:00:00, 2015-01-25 00:00:00, 201...",491,2015-05-24,2015-05-24
9,ARI,HU,data_range,524,66,458,5,"[418, 19, 1, 1, 19]","[2014-10-05 00:00:00, 2023-05-28 00:00:00, 202...",418,2014-10-05,2024-05-26
6,ARI,ES,data_range,524,158,366,1,[366],[2014-10-05 00:00:00],366,2014-10-05,2014-10-05
2,ARI,CY,data_range,524,71,453,4,"[88, 1, 1, 363]","[2015-10-04 00:00:00, 2017-07-02 00:00:00, 201...",363,2017-11-05,2017-11-05
8,ARI,FR,data_range,524,167,357,5,"[304, 15, 11, 13, 14]","[2014-10-05 00:00:00, 2021-04-25 00:00:00, 202...",304,2014-10-05,2024-04-28
15,ARI,PT,data_range,524,124,400,9,"[280, 3, 28, 1, 1, 1, 1, 1, 84]","[2014-10-05 00:00:00, 2020-02-23 00:00:00, 202...",280,2014-10-05,2023-03-12
21,ILI,CY,data_range,524,242,282,5,"[1, 2, 1, 1, 277]","[2017-02-12 00:00:00, 2017-05-28 00:00:00, 201...",277,2019-06-30,2019-06-30
25,ILI,ES,data_range,524,199,325,6,"[19, 19, 19, 19, 19, 230]","[2015-05-24 00:00:00, 2016-05-29 00:00:00, 201...",230,2020-05-24,2020-05-24
14,ARI,NL,data_range,524,369,155,4,"[18, 20, 11, 106]","[2014-10-05 00:00:00, 2015-02-15 00:00:00, 201...",106,2022-10-09,2022-10-09
41,ILI,PT,data_range,524,435,89,6,"[1, 1, 1, 1, 1, 84]","[2018-07-29 00:00:00, 2020-12-27 00:00:00, 202...",84,2023-03-12,2023-03-12
